# 08 — Connection-aware search

**CPU runtime → Runtime → Run all → upload your completed notebook 07 results ZIP.**

This runs a new, paired development experiment on the same 32 malicious records and two frozen IDS models. It compares native CAA with repair after search and repair inside MOEVA. Every reported method uses the same final validity filter.

Two necessary conditions are fixed before these runs: positive aggregate sums require positive maxima; positive duration requires recorded endpoints in the same bucket. The second rule addresses the duration-only candidate found in notebook 07.

Keep your earlier ZIPs and notebooks. Every session and export here is new. No model training or packet replay occurs.

## 1. Create a new session and upload notebook 07

Leave `INPUT_ZIP` empty for the upload button. Alternatively, provide its absolute runtime path. Upload the completed **07** ZIP, not the older 06 ZIP. The notebook restores the frozen inputs/scaler and generates fresh attacks.

In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

INPUT_ZIP = "" #@param {type:"string"}
INPUT_ZIP = os.environ.get("PHD_INPUT_ZIP", INPUT_ZIP)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
if os.environ.get("PHD_NO_DOWNLOAD") == "1":
    DOWNLOAD_RESULTS = False
REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_ConnectionSearch")).resolve()
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} is not a git checkout. Choose a new workspace.")
for required in ["scripts/bootstrap.py", "baseline/run_connection_search.py", "notebooks/08_Connection_Aware_Search.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the updated repository.")

# Existing checkouts are preserved; no pull/reset/checkout is performed.
def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("connection_search")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ["baseline", "scripts", "tests"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/08_Connection_Aware_Search.ipynb", ROOT / "docs/CONNECTION_AWARE_SEARCH_GUIDE.md"]
context = {
    "project_repository": REPO_URL,
    "project_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    "experiment": "connection_aware_evolutionary_search", "evaluation_device": "cpu",
}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Export partial logs with the last cell.")

print("Project commit:", context["project_commit"])
print("New session:", SESSION)

In [ ]:
if INPUT_ZIP.strip():
    INPUT = Path(INPUT_ZIP).expanduser().resolve()
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_ZIP to your completed notebook 07 results ZIP.")
    upload_dir = SESSION / "upload"
    upload_dir.mkdir(exist_ok=False)
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one completed notebook 07 results ZIP.")
    INPUT = upload_dir / next(iter(uploaded))
    del uploaded
if not INPUT.is_file() or INPUT.suffix.lower() != ".zip":
    raise ValueError("INPUT_ZIP must identify an existing ZIP file.")
print("Input evidence:", INPUT.name)
print("Input SHA256:", hashlib.sha256(INPUT.read_bytes()).hexdigest())

## 2. Prepare the pinned environment

The isolated baseline uses Python 3.8.20, NumPy 1.23.5 and PyTorch 1.12.1+cpu. The notebook kernel remains unchanged. Use a CPU runtime for this paired experiment; a Colab GPU will not accelerate this pinned implementation.

A fresh setup downloads approximately 348 MB of feature data plus checkpoints. Only native training/development rows are parsed for control checks. The inherited frozen scaler used all feature rows originally, so this is not a leakage-free final test.

In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
setup_copy = SESSION / unique_name("setup_report")
shutil.copy2(ROOT / ".cache/setup_report.json", setup_copy.with_suffix(".json"))
print("Pinned experiment interpreter:", PYTHON)

In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
manifest_copy = SESSION / (unique_name("download_manifest") + ".json")
shutil.copy2(ROOT / "assets/download_manifest.json", manifest_copy)

## 3. Run the paired experiment

The runner first verifies the input, pinned assets and 143,046 clean controls in float64 and float32. It then runs 18 fresh CAA attacks: two models × three seeds × original generation/two in-loop variants. Each uses the original 10 CAPGD steps, 100 MOEVA generations, 50 offspring and scaled L2 budget 0.5.

Five comparisons per model/seed:
- `caa_filter`: native generation with the common final filter.
- `caa_profile_budget_end`: budget/profile repair after native generation.
- `caa_profile_relation_end`: relationship/budget/profile repair after native generation.
- `caa_profile_budget_inloop`: budget/profile repair before MOEVA fitness.
- `caa_profile_relation_inloop`: relationship/budget/profile repair before MOEVA fitness.

Pre-MOEVA inputs and core model-forward counts must match the fresh original run. A mismatch stops the experiment. Repair adds work; equal core counts do not mean equal total computation or runtime.

**Scope:** hard repair is inside MOEVA. CAPGD inner iterations remain native; its output faces the added CAA stage selector and final gate. Keep this cell running until completion; runtime depends on the CPU allocation.

In [ ]:
RUN = SESSION / "experiment"
run_command([PYTHON, ROOT / "baseline/run_connection_search.py",
             "--input-zip", INPUT, "--out", RUN,
             "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets"], "experiment.log")
print("Completed experiment:", RUN)

## 4. Independently review exported evidence

This second program derives feature pairs independently and rechecks every saved source and final vector using the native checker and frozen models. It reconstructs pool counts and verifies selected-vector provenance. It also checks matched budgets and repair/evaluation digest logs. Intermediate optimizer vectors are not exported, so they cannot be independently reconstructed from this archive.

In [ ]:
REVIEW = SESSION / "independent_review.json"
run_command([PYTHON, ROOT / "scripts/review_notebook08_evidence.py", RUN, REVIEW], "review.log")
assert json.loads(REVIEW.read_text())["status"] == "passed"

## 5. Read the result

`newly_evaded` counts initially detected malicious originals that have a changed, valid adversarial representative. Its denominator is `clean_detected`, not all traffic. Four of the full pilot's 32 originals were already missed; these do not count as new evasions.

`covered_records` measures whether any changed, valid candidate was available, whether or not it evaded detection. More coverage alone does not establish a better attack. `native_original_reference.csv` separately retains the native returned result before the stronger filter.

Do not report these small development counts as real-world accuracy or proof that the IDS is robust.

In [ ]:
import pandas as pd
from IPython.display import display

protocol = json.loads((RUN / "protocol.json").read_text())
results = json.loads((RUN / "results.json").read_text())
review = json.loads(REVIEW.read_text())
assert protocol["status"] == results["status"] == "completed"
assert not protocol["smoke_check_only"] and not protocol["test_partition_evaluated"]
print("Clean training/development controls:")
display(pd.read_csv(RUN / "clean_controls.csv"))
comparison = pd.read_csv(RUN / "comparison.csv")
display(comparison[["model", "seed", "method", "clean_detected", "covered_records",
                    "pool_evaded_records", "newly_evaded", "new_evasion_rate",
                    "generation_forward_rows", "generation_forward_counts_match_original"]])
print("Evasions by seed (same originals, not independent traffic samples):")
display(comparison.pivot(index=["model", "method"], columns="seed", values="newly_evaded"))
print("Independent review:", review["status"], review["counts"])
print("Experiment runtime (minutes):", round(results["wall_seconds"] / 60, 2))
print("Passing necessary feature conditions does not prove realizable or still-malicious traffic.")

## 6. Download and share the new ZIP

The export contains the input evidence, fixed rule profile, clean-control audit, all exported stage candidates and source masks, final selections, optimizer logs, independent review, code snapshot and runtime logs. Dataset files, weights and the environment are excluded.

Download this new ZIP and share it for analysis. If a cell failed, run this export cell separately to preserve the partial evidence. It never overwrites an earlier ZIP.

In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
import zipfile
archive = export_base.with_suffix(".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SESSION.rglob("*")):
        if path.is_file() and "upload" not in path.relative_to(SESSION).parts:
            z.write(path, path.relative_to(SESSION.parent))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))

## Research interpretation

This experiment asks whether enforcing necessary traffic-feature conditions during search produces more valid evasions than filtering or repairing afterward at the same core attack budget. An unchanged or worse result is useful evidence and must be retained.

The rules were motivated by earlier development observations. They do not constitute a novel defense or complete packet realizability. The 32 selected malicious records and three repeated seeds cannot support overall accuracy or population-level robustness claims. Stronger claims require additional development replication and a separately designed, locked evaluation.